<a href="https://colab.research.google.com/github/MatthewPena2/AI-vs-Real-Image-Analyzer/blob/main/EfficientNetB0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os # used to store kaggle credentials
from google.colab import userdata # obtains API key and account username stored in google colab

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Very quick check to ensure the cell executed the lines above
print("Kaggle credentials obtained successfully!")

Kaggle credentials obtained successfully!


In [2]:
# Install Kaggle and download CIFAKE
# Install kaggle library
!pip install -q kaggle

# Download CIFAKE using the full dataset path
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images

# Unzip the .zip file where the dataset is stored
# "*.zip" allows us to unzip the dataset file without specifying a name
!unzip -q "*.zip" -d cifake_data

# Very quick check to ensure the cell executed the lines above
# There should be a popup showing the files have been downloaded
print("Dataset download and unzip was successful!")

Dataset URL: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
License(s): other
100% 105M/105M [00:06<00:00, 17.2MB/s]

Dataset download and unzip was successful!


In [3]:
import torch # import PyTortch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ------Preprocessing------
# Define image transformations for EfficientNet-B0
# This model expects 224x224 input image sizes and standard ImageNet normalization)
transform = transforms.Compose([
    transforms.Resize((224, 224)), # Resizes CIFAKE's small 32 x 32 image size to fit EfficientNet-B0 requirements
    transforms.RandomHorizontalFlip(p=0.5), # Lightweight data augmentation - prevents overfitting
    transforms.ToTensor(), # Loaded Python pixel values are stored as ints; PyTorch models require floating-point tensors
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalizes an iamge to match ImageNet's basic color scales
])

# Directory Paths
train_dir = 'cifake_data/train'
test_dir = 'cifake_data/test'

# Load Datasets using ImageLoader - maps folder names to class labels
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)

# Create DataLoaders for batching during training/testing
# Uses deep learning standard batch size of 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Quick check to ensure program executed to this point
print(f"Dataset pipeline succesfully initiated")
print(f"Classes found: {train_dataset.classes}") # Check to see if PyTorch succesfully scanned folders
print(f"Total training batches: {len(train_loader)}") # Check to see training batch size
print(f"Total test batches: {len(test_loader)}") # Check to see test batch size

# ~83.33% Training : ~16.67% Test
# With a batch size of 32, 3,125 batches = 100,000 training images
# With a batch size of 32, 625 batches = 20,000 test images
# This split matches the standard CIFAKE test/training split

Dataset pipeline succesfully initiated
Classes found: ['FAKE', 'REAL']
Total training batches: 3125
Total test batches: 625


In [6]:
import torch

# Device Setup - Check to see if T4 GPU is in use
# CUDA is the hardware-accelerated NVIDIA GPU (provided to Google Colab for free)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
  print(f"GPU Model: {torch.cuda.get_device_name(0)}")


Using device: cuda
GPU Model: Tesla T4
